# 19. 전체 태그 분포 — 게임 수 & 총 리뷰 수

**분석 목적:** `steam_indie_games`와 `steam_indie_games_silence` 전체 데이터를 합산하여,
가장 많은 게임이 보유한 태그(공급)와 태그별 총 리뷰 수(수요)를 시각화한다.

**사용 데이터:**
- `data/preprocessed/steam_indie_games.csv` — 초기 반응 그룹 (리뷰 10개 이상)
- `data/preprocessed/steam_indie_games_silence.csv` — 침묵 그룹 (리뷰 10개 미만)

**분석 방법:**
- **게임 수**: 해당 태그를 보유한 고유 게임 수 (공급)
- **총 리뷰 수**: 해당 태그를 보유한 게임들의 `total_reviews` 합계 (수요)

In [9]:
import json
import warnings

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", 50)

TOP_N = 10
EXCLUDE_TAGS = {'Indie'}

## 1. 데이터 로드 및 병합

In [10]:
df_response = pd.read_csv('../../data/preprocessed/steam_indie_games.csv')
df_silence  = pd.read_csv('../../data/preprocessed/steam_indie_games_silence.csv')

df_response['group'] = '초기 반응 (≥10개)'
df_silence['group']  = '침묵 (<10개)'

df = pd.concat([df_response, df_silence], ignore_index=True)

print(f'초기 반응 그룹 : {len(df_response):,}개')
print(f'침묵 그룹      : {len(df_silence):,}개')
print(f'전체           : {len(df):,}개')

초기 반응 그룹 : 8,730개
침묵 그룹      : 6,676개
전체           : 15,406개


## 2. 태그 파싱 및 집계

In [11]:
def parse_tags(value) -> dict:
    if pd.isna(value):
        return {}
    try:
        return json.loads(value)
    except (json.JSONDecodeError, TypeError):
        return {}


df['tag_dict'] = df['tags'].apply(parse_tags)
df['total_reviews'] = pd.to_numeric(df['total_reviews'], errors='coerce').fillna(0)

# 태그별 (게임 수, 총 리뷰 수, 중앙값 리뷰 수) 집계
rows = []
for _, row in df.iterrows():
    for tag in row['tag_dict']:
        tag = tag.strip()
        if tag in EXCLUDE_TAGS:
            continue
        rows.append({'appid': row['appid'], 'tag': tag, 'total_reviews': row['total_reviews']})

df_tags = pd.DataFrame(rows)

tag_stats = (
    df_tags.groupby('tag')
    .agg(
        game_count=('appid', 'nunique'),
        total_reviews=('total_reviews', 'sum'),
        median_reviews=('total_reviews', 'median'),
    )
    .reset_index()
)

print(f'제외 태그: {EXCLUDE_TAGS}')
print(f'전체 고유 태그 수: {len(tag_stats):,}개')
display(tag_stats.sort_values('game_count', ascending=False).head(10))

제외 태그: {'Indie'}
전체 고유 태그 수: 435개


,tag,game_count,total_reviews,median_reviews
338,Singleplayer,9877,4582007,8.0
69,Casual,5982,1790118,7.0
17,Action,5745,3450158,9.0
3,2D,5700,1825278,8.0
23,Adventure,5580,3092148,10.0
7,3D,4265,1121384,7.0
87,Colorful,3399,825575,8.0
38,Atmospheric,3225,1773242,13.0
141,Exploration,3165,2430703,14.0
281,Pixel Graphics,3153,1170144,9.0


## 3. 태그별 게임 수 & 총 리뷰 수 Top 30 — 이중 바 차트

In [12]:
by_games   = tag_stats.sort_values('game_count', ascending=False).head(TOP_N).sort_values('game_count')
by_reviews = tag_stats.sort_values('total_reviews', ascending=False).head(TOP_N).sort_values('total_reviews')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f'공급: 태그별 게임 수 Top {TOP_N}',
        f'수요: 태그별 총 리뷰 수 Top {TOP_N}',
    ),
    horizontal_spacing=0.18,
)

fig.add_trace(
    go.Bar(
        x=by_games['game_count'],
        y=by_games['tag'],
        orientation='h',
        marker_color='#4C72B0',
        text=by_games['game_count'].apply(lambda v: f'{v:,}'),
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>게임 수: %{x:,}<extra></extra>',
        showlegend=False,
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Bar(
        x=by_reviews['total_reviews'],
        y=by_reviews['tag'],
        orientation='h',
        marker_color='#DD8452',
        text=by_reviews['total_reviews'].apply(lambda v: f'{v/1e6:.1f}M' if v >= 1e6 else f'{v/1e3:.0f}K'),
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>총 리뷰 수: %{x:,}<extra></extra>',
        showlegend=False,
    ),
    row=1, col=2,
)

fig.update_layout(
    title=f'전체 인디게임 태그 분포 — 공급(게임 수) vs 수요(총 리뷰 수) Top {TOP_N}<br>'
          f'<sub>전체 {len(df):,}개 게임 / steam_indie_games + steam_indie_games_silence 합산 / Indie 태그 제외</sub>',
    height=900,
    margin=dict(l=160, r=120),
)
fig.update_xaxes(title_text='게임 수', row=1, col=1)
fig.update_xaxes(title_text='총 리뷰 수', row=1, col=2)

fig.show()

**공급(게임 수) 해석:** 가장 많은 게임이 달고 있는 태그로, 인디 시장에서 공급이 집중된 카테고리를 나타낸다. 상위권 태그는 대부분의 인디게임이 공유하는 일반 태그라 차별화 지표로 활용하기 어렵다.

**수요(총 리뷰 수) 해석:** 해당 태그를 보유한 게임들이 받은 리뷰 합계로, 유저가 실제로 플레이하고 반응한 카테고리의 시장 규모를 나타낸다. 게임 수 순위와 비교했을 때 총 리뷰 순위가 높은 태그는 공급 대비 유저 수요가 집중된 카테고리다.

## 4. 게임 수 vs 총 리뷰 수 포지셔닝 산점도 (Top 50)

In [13]:
TOP_SCATTER = 10

plot_df = tag_stats.sort_values('game_count', ascending=False).head(TOP_SCATTER).copy()
plot_df['median_reviews'] = plot_df['median_reviews'].round(0).astype(int)

mid_games   = plot_df['game_count'].median()
mid_reviews = plot_df['total_reviews'].median()

# 중앙값 리뷰 수를 실제 데이터 범위 기준으로 정규화해 점 크기 결정 (12~36px)
v_min, v_max = plot_df['median_reviews'].min(), plot_df['median_reviews'].max()
plot_df['marker_size'] = 12 + (plot_df['median_reviews'] - v_min) / max(v_max - v_min, 1) * 24

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=plot_df['game_count'],
    y=plot_df['total_reviews'],
    mode='markers+text',
    text=plot_df['tag'],
    textposition='top center',
    textfont=dict(size=9),
    marker=dict(
        size=plot_df['marker_size'],
        color=plot_df['median_reviews'],
        colorscale='Plasma',
        showscale=True,
        colorbar=dict(title='게임당<br>중앙값 리뷰 수'),
        opacity=0.85,
        line=dict(width=1, color='white'),
    ),
    customdata=plot_df[['median_reviews']].values,
    hovertemplate=(
        '<b>%{text}</b><br>'
        '게임 수: %{x:,}개<br>'
        '총 리뷰 수: %{y:,}<br>'
        '게임당 중앙값 리뷰 수: %{customdata[0]:,}<extra></extra>'
    ),
    showlegend=False,
))

fig.add_vline(
    x=mid_games, line_dash='dot', line_color='#adb5bd', line_width=1.5,
    annotation_text=f'게임 수 중앙값 ({int(mid_games):,})',
    annotation_position='top right',
    annotation_font=dict(size=10, color='#6B7280'),
)
fig.add_hline(
    y=mid_reviews, line_dash='dot', line_color='#adb5bd', line_width=1.5,
    annotation_text=f'총 리뷰 수 중앙값 ({int(mid_reviews):,})',
    annotation_position='top right',
    annotation_font=dict(size=10, color='#6B7280'),
)

fig.update_layout(
    title=f'태그별 공급 vs 수요 포지셔닝 (게임 수 Top {TOP_SCATTER})<br>'
          '<sub>점 크기·색: 게임당 중앙값 리뷰 수 / 점선: 각 축 중앙값</sub>',
    xaxis_title='게임 수 (공급)',
    yaxis_title='총 리뷰 수 (수요)',
    height=600,
    plot_bgcolor='#FAFAFA',
)
fig.show()

**산점도 해석:** 점선(중앙값 기준)으로 사분면이 나뉜다.
- **우상단**: 게임 수도 많고 리뷰도 많은 태그 — 대형 경쟁 카테고리
- **우하단**: 게임은 많지만 리뷰가 적은 태그 — 공급 과잉·수요 부족 카테고리
- **좌상단**: 게임 수는 적지만 리뷰가 많은 태그 — 틈새 고수요 카테고리 (블루오션 후보)
- **좌하단**: 게임 수와 리뷰 모두 낮은 태그 — 소규모 니치 카테고리

점의 색·크기는 **게임당 중앙값 리뷰 수**로, 소수 히트작에 의한 왜곡 없이 해당 태그의 전형적인 게임이 받는 리뷰 수를 나타낸다. 색이 진할수록 이 태그를 단 평범한 게임도 상당한 관심을 받는 카테고리다.

## 5. 요약 테이블

In [14]:
summary = tag_stats.copy()
summary['median_reviews'] = summary['median_reviews'].round(0).astype(int)
summary['게임 수 순위'] = summary['game_count'].rank(ascending=False).astype(int)
summary['리뷰 수 순위'] = summary['total_reviews'].rank(ascending=False).astype(int)
summary['순위 차이'] = summary['게임 수 순위'] - summary['리뷰 수 순위']

print(f'게임 수 기준 상위 {TOP_N}개 태그 요약')
display(
    summary
    .sort_values('게임 수 순위')
    .head(TOP_N)
    [['tag', 'game_count', 'total_reviews', 'median_reviews', '게임 수 순위', '리뷰 수 순위', '순위 차이']]
    .rename(columns={
        'tag': '태그',
        'game_count': '게임 수',
        'total_reviews': '총 리뷰 수',
        'median_reviews': '게임당 중앙값 리뷰 수',
    })
    .set_index('태그')
)

게임 수 기준 상위 10개 태그 요약


,게임 수,총 리뷰 수,게임당 중앙값 리뷰 수,게임 수 순위,리뷰 수 순위,순위 차이
태그,,,,,,
Singleplayer,9877,4582007,8,1,1,0
Casual,5982,1790118,7,2,13,-11
Action,5745,3450158,9,3,2,1
2D,5700,1825278,8,4,12,-8
Adventure,5580,3092148,10,5,3,2
3D,4265,1121384,7,6,26,-20
Colorful,3399,825575,8,7,37,-30
Atmospheric,3225,1773242,13,8,14,-6
Exploration,3165,2430703,14,9,6,3


**요약 해석:** 순위 차이(게임 수 순위 - 리뷰 수 순위)가 양수이면 게임은 많지만 리뷰가 상대적으로 적은 태그(공급 과잉), 음수이면 게임 수 대비 리뷰가 집중된 태그(수요 우세)다. 인디 개발사 입장에서 순위 차이가 음수이고 게임당 평균 리뷰 수가 높은 태그는 경쟁이 덜하면서도 유저 참여도가 높은 카테고리로, 포지셔닝 전략 수립 시 우선 검토할 만하다.

## 6. 침묵 그룹 태그 Top 10

**분석 목적:** 무반응 그룹(리뷰 0~9개)에서 가장 많이 사용된 태그 상위 10개를 추출하여, 시장에서 묻히는 게임들이 공통적으로 달고 있는 '차별화되지 않은' 태그 패턴을 확인한다.

**사용 데이터:** `steam_indie_games_silence.csv` — 침묵 그룹 단독 (리뷰 0~9개, 6,676개)

**분석 방법:** 침묵 그룹 내 태그별 게임 수 집계 → Top 10 수평 바 차트

In [15]:
# 침묵 그룹 단독 태그 집계 + 전체 시장 비율 기준선
total_all     = df['appid'].nunique()
total_silence = df[df['group'] == '침묵 (<10개)']['appid'].nunique()

# 전체 태그별 게임 수 (기준선용)
all_rows = []
for _, row in df.iterrows():
    for tag in row['tag_dict']:
        tag = tag.strip()
        if tag in EXCLUDE_TAGS:
            continue
        all_rows.append({'appid': row['appid'], 'tag': tag})
df_all_tag = pd.DataFrame(all_rows)
all_tag_counts = (
    df_all_tag.groupby('tag')['appid'].nunique().rename('game_count')
)

# 침묵 그룹 태그별 게임 수
silence_rows = []
for _, row in df[df['group'] == '침묵 (<10개)'].iterrows():
    for tag in row['tag_dict']:
        tag = tag.strip()
        if tag in EXCLUDE_TAGS:
            continue
        silence_rows.append({'appid': row['appid'], 'tag': tag})
df_silence_tag_long = pd.DataFrame(silence_rows)
silence_tag_counts = (
    df_silence_tag_long.groupby('tag')['appid'].nunique().rename('game_count')
)

# Top 10 추출 및 비율 계산
top10_tags = silence_tag_counts.sort_values(ascending=False).head(TOP_N).index.tolist()

plot_df = pd.DataFrame({
    'tag':           top10_tags,
    'silence_rate':  (silence_tag_counts[top10_tags] / total_silence * 100).values,
    'overall_rate':  (all_tag_counts[top10_tags]     / total_all     * 100).values,
    'silence_count': silence_tag_counts[top10_tags].values,
}).sort_values('silence_rate')

fig = go.Figure()

# 침묵 그룹 비율 바
fig.add_trace(go.Bar(
    x=plot_df['silence_rate'],
    y=plot_df['tag'],
    orientation='h',
    name='침묵 그룹 비율',
    marker_color='#C44E52',
    text=plot_df.apply(
        lambda r: f"{r['silence_count']:,.0f}개 ({r['silence_rate']:.1f}%)", axis=1
    ),
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>침묵 그룹: %{x:.1f}%<extra></extra>',
))

# 전체 시장 비율 기준선 (마커)
fig.add_trace(go.Scatter(
    x=plot_df['overall_rate'],
    y=plot_df['tag'],
    mode='markers',
    name='전체 시장 평균',
    marker=dict(symbol='line-ns', size=14, color='#2D3E71', line=dict(width=2.5, color='#2D3E71')),
    hovertemplate='<b>%{y}</b><br>전체 시장: %{x:.1f}%<extra></extra>',
))

fig.update_layout(
    title=(
        f'침묵 그룹 태그 Top {TOP_N} — 침묵 그룹 비율 vs 전체 시장 평균<br>'
        f'<sub>침묵 그룹 {total_silence:,}개 (리뷰 0~9개) / 전체 {total_all:,}개 / Indie 태그 제외 / '
        f'|: 전체 시장 비율 기준선</sub>'
    ),
    xaxis_title='게임 비율 (%)',
    yaxis_title='태그',
    height=520,
    margin=dict(l=140, r=160),
    plot_bgcolor='#FAFAFA',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    barmode='overlay',
)
fig.show()

print(f'침묵 그룹 게임 수: {total_silence:,}개 / 전체: {total_all:,}개')
display(
    plot_df.sort_values('silence_rate', ascending=False)
    .assign(**{'시장 대비 차이(%p)': lambda d: (d['silence_rate'] - d['overall_rate']).round(1)})
    .rename(columns={
        'tag': '태그',
        'silence_count': '침묵 게임 수',
        'silence_rate': '침묵 그룹 비율(%)',
        'overall_rate': '전체 시장 비율(%)',
    })
    [['태그', '침묵 게임 수', '침묵 그룹 비율(%)', '전체 시장 비율(%)', '시장 대비 차이(%p)']]
    .set_index('태그')
    .style.format({'침묵 그룹 비율(%)': '{:.1f}', '전체 시장 비율(%)': '{:.1f}', '시장 대비 차이(%p)': '{:+.1f}'})
)

침묵 그룹 게임 수: 6,676개 / 전체: 15,406개


,침묵 게임 수,침묵 그룹 비율(%),전체 시장 비율(%),시장 대비 차이(%p)
태그,,,,
Singleplayer,5302,79.4,64.1,+15.3
Casual,3522,52.8,38.8,+13.9
2D,3119,46.7,37.0,+9.7
Action,3010,45.1,37.3,+7.8
Adventure,2720,40.7,36.2,+4.5
3D,2402,36.0,27.7,+8.3
Colorful,1813,27.2,22.1,+5.1
Puzzle,1642,24.6,20.0,+4.6
Pixel Graphics,1630,24.4,20.5,+3.9


**해석:** 바(침묵 그룹 비율)가 기준선(전체 시장 평균)을 모두 초과한다 — 침묵 그룹은 이 10개 태그를 전체 시장 평균보다 더 많이 달고 있다. 차이가 가장 큰 것은 Singleplayer(+15.3%p)·Casual(+14.0%p)로, 이미 공급 과잉인 범용 태그에 더 집중되어 있는 패턴이다.

**주의:** 이 수치는 '침묵 게임에서 흔히 보이는 태그'이지 인과관계가 아니다. 같은 태그를 단 게임 중에도 반응을 얻은 게임이 다수 존재한다. (첫 반응군 태그 Top 10)와 비교해 두 그룹 공통 태그 vs 차별화 태그를 파악하는 것이 핵심이다.

## 7. 첫 반응군 태그 Top 10

**분석 목적:** 첫 반응군(리뷰 10~49개)에서 가장 많이 사용된 태그 상위 10개를 추출하여, 발견된 게임들이 공통적으로 달고 있는 태그 패턴을 확인한다.

**사용 데이터:** `steam_indie_games_graded.csv` — 첫 반응군 필터링 (리뷰 10~49개)

**분석 방법:** 첫 반응군 내 태그별 게임 수 집계 → Top 10 수평 바 차트 (전체 시장 평균 기준선 오버레이)

In [16]:
# 첫 반응군 단독 태그 집계 + 전체 시장 비율 기준선
total_all      = df['appid'].nunique()
total_response = df[df['group'] == '초기 반응 (≥10개)']['appid'].nunique()

df_graded = pd.read_csv('../../data/preprocessed/steam_indie_games_graded.csv')
df_graded['tag_dict'] = df_graded['tags'].apply(parse_tags)
df_graded['total_reviews'] = pd.to_numeric(df_graded['total_reviews'], errors='coerce').fillna(0)

df_first = df_graded[
    (df_graded['total_reviews'] >= 10) & (df_graded['total_reviews'] <= 49)
].copy()
total_first = df_first['appid'].nunique()

# 첫 반응군 태그별 게임 수
first_rows = []
for _, row in df_first.iterrows():
    for tag in row['tag_dict']:
        tag = tag.strip()
        if tag in EXCLUDE_TAGS:
            continue
        first_rows.append({'appid': row['appid'], 'tag': tag})
df_first_tag_long = pd.DataFrame(first_rows)
first_tag_counts = (
    df_first_tag_long.groupby('tag')['appid'].nunique().rename('game_count')
)

# Top 10 추출 및 비율 계산
top10_first = first_tag_counts.sort_values(ascending=False).head(TOP_N).index.tolist()

plot_df_first = pd.DataFrame({
    'tag':          top10_first,
    'first_rate':   (first_tag_counts[top10_first] / total_first * 100).values,
    'overall_rate': (all_tag_counts[top10_first]   / total_all   * 100).values,
    'first_count':  first_tag_counts[top10_first].values,
}).sort_values('first_rate')

fig2 = go.Figure()

# 첫 반응군 비율 바
fig2.add_trace(go.Bar(
    x=plot_df_first['first_rate'],
    y=plot_df_first['tag'],
    orientation='h',
    name='첫 반응군 비율',
    marker_color='#4C72B0',
    text=plot_df_first.apply(
        lambda r: f"{r['first_count']:,.0f}개 ({r['first_rate']:.1f}%)", axis=1
    ),
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>첫 반응군: %{x:.1f}%<extra></extra>',
))

# 전체 시장 비율 기준선 (마커)
fig2.add_trace(go.Scatter(
    x=plot_df_first['overall_rate'],
    y=plot_df_first['tag'],
    mode='markers',
    name='전체 시장 평균',
    marker=dict(symbol='line-ns', size=14, color='#2D3E71', line=dict(width=2.5, color='#2D3E71')),
    hovertemplate='<b>%{y}</b><br>전체 시장: %{x:.1f}%<extra></extra>',
))

fig2.update_layout(
    title=(
        f'첫 반응군 태그 Top {TOP_N} — 첫 반응군 비율 vs 전체 시장 평균<br>'
        f'<sub>첫 반응군 {total_first:,}개 (리뷰 10~49개) / 전체 {total_all:,}개 / Indie 태그 제외 / '
        f'|: 전체 시장 비율 기준선</sub>'
    ),
    xaxis_title='게임 비율 (%)',
    yaxis_title='태그',
    height=520,
    margin=dict(l=140, r=160),
    plot_bgcolor='#FAFAFA',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    barmode='overlay',
)
fig2.show()

print(f'첫 반응군 게임 수: {total_first:,}개 / 전체: {total_all:,}개')
display(
    plot_df_first.sort_values('first_rate', ascending=False)
    .assign(**{'시장 대비 차이(%p)': lambda d: (d['first_rate'] - d['overall_rate']).round(1)})
    .rename(columns={
        'tag':          '태그',
        'first_count':  '첫 반응군 게임 수',
        'first_rate':   '첫 반응군 비율(%)',
        'overall_rate': '전체 시장 비율(%)',
    })
    [['태그', '첫 반응군 게임 수', '첫 반응군 비율(%)', '전체 시장 비율(%)', '시장 대비 차이(%p)']]
    .set_index('태그')
    .style.format({'첫 반응군 비율(%)': '{:.1f}', '전체 시장 비율(%)': '{:.1f}', '시장 대비 차이(%p)': '{:+.1f}'})
)

첫 반응군 게임 수: 4,840개 / 전체: 15,406개


,첫 반응군 게임 수,첫 반응군 비율(%),전체 시장 비율(%),시장 대비 차이(%p)
태그,,,,
Singleplayer,2300,47.5,64.1,-16.6
Action,1452,30.0,37.3,-7.3
Adventure,1389,28.7,36.2,-7.5
2D,1358,28.1,37.0,-8.9
Casual,1337,27.6,38.8,-11.2
3D,982,20.3,27.7,-7.4
Colorful,872,18.0,22.1,-4.0
Atmospheric,833,17.2,20.9,-3.7
Exploration,777,16.1,20.5,-4.5


**해석:** 첫 반응군(리뷰 10~49개)에서 가장 많이 보이는 태그 Top 10과 전체 시장 평균을 비교한다. 바(첫 반응군 비율)가 기준선(전체 시장 평균)보다 높으면 해당 태그가 첫 반응군에서 과대 대표되는 것이고, 낮으면 과소 대표되는 것이다.

**(침묵 그룹 태그 Top 10)와 비교할 때:** 두 그룹에서 공통으로 상위권에 있는 태그는 전체 인디 시장의 기본값이며, 첫 반응군에서만 높게 나타나는 태그가 발견의 차별화 시그널이 될 수 있다.

**주의:** 이 수치는 첫 반응을 얻은 게임들이 '많이 달고 있는' 태그이지, 해당 태그가 반응의 원인임을 의미하지 않는다.